# Hypothesis Testing & Feature Exploration

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import category_encoders as ce
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import ElasticNet
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

## Load Data

In [2]:
patterns = pd.read_pickle('preprocessed_patterns.pkl')

In [3]:
patterns = patterns.drop(columns=['pattern_id', 'projects_count', 'favorites_count', 'author_id', 'estimated_revenue', 'total_languages', 'yarn_ids'])

In [4]:
patterns.describe()

,queued_projects_count,yardage,previously_published_patterns,days_since_publication,projects_per_day,price_usd,days_since_previous_pattern
count,559308.000000,559308.000000,559308.000000,559308.000000,559308.000000,559308.000000,559308.000000
mean,28.099493,412.522251,105.490880,2506.304778,0.008950,5.339493,1440.806184
std,139.963785,1064.542127,277.811141,1619.530390,0.138536,2.325749,1437.530544
min,0.000000,0.000000,1.000000,33.000000,0.000000,0.001550,0.000000
25%,1.000000,0.000000,10.000000,1169.000000,0.000000,4.000000,306.000000
50%,4.000000,197.000000,31.000000,2261.000000,0.000591,5.000000,986.000000
75%,15.000000,550.000000,88.000000,3698.000000,0.003049,6.474820,2165.000000
max,16025.000000,428696.000000,5009.000000,6989.000000,55.542169,100.000000,6910.000000


## Prepare Data

In [ ]:
# Convert yarn_weight to numeric before scaling
patterns['yarn_weight'] = patterns['yarn_weight'].map({
	'Thread': 1,
	'Cobweb': 2,
	'Lace': 3,
	'Light Fingering': 4,
	'Fingering': 5,
	'Sport': 6,
	'DK': 7,
	'Worsted': 8,
	'Aran': 9,
	'Bulky': 10,
	'Super Bulky': 11,
	'Jumbo': 12
}).fillna(8.5)
# Will be log-transformed & scaled along with some very skewed features; so needs to be exponentialized for the log to return correctly
patterns['yarn_weight'] = np.exp(patterns['yarn_weight'])

# Get log of then scale non-dummy features
float_features = ['yarn_weight', 'queued_projects_count', 'yardage', 'previously_published_patterns', 'days_since_publication', 'projects_per_day', 'price_usd', 'days_since_previous_pattern']
for feature in float_features:
    patterns[feature] = np.log1p(patterns[feature])

scaler = StandardScaler()
patterns[float_features] = scaler.fit_transform(patterns[float_features])

# Drop outliers with generous threshold of 4 standard deviations
for feature in float_features:
	patterns = patterns[np.abs(patterns[feature]) <= 4]

In [6]:
# Rename has_uk_terminology and has_us_terminology to uk_terminology and us_terminology for clarity
patterns = patterns.rename(columns={'has_uk_terminology': 'uk_terminology', 'has_us_terminology': 'us_terminology'})

In [7]:
# Where craft == knitting, set new `knit` column to 1
patterns['knit'] = ((patterns['craft'] == 'Knitting')).astype(int)

# Drop redundant/unhelpful columns
patterns = patterns.drop(columns=['craft', 'attributes_', 'pattern_source_type_names_', 'queued_projects_count', 'pattern_author'])

In [8]:
# Create weight column to weigh more recent patterns more heavily, using an exponential decay function with half-life of 4 years
half_life = 1460
lambda_ = np.log(2) / half_life
patterns['weight'] = np.exp(-lambda_ * patterns['days_since_publication'])

patterns = patterns.drop(columns = ['days_since_publication'])

In [9]:
# Convert `season` values to lowercase
patterns['season'] = patterns['season'].str.lower()

# Convert `season` to dummy
patterns = pd.get_dummies(patterns, columns=['season'], drop_first=True)

In [10]:
# Set final_category to concatenated string of supercategory, category, subcategory, and babycategory (in that order) with underscores as separators
# Skip missing values when concatenating
patterns['final_category'] = patterns['supercategory'].fillna('') + '_' + patterns['category'].fillna('') + '_' + patterns['subcategory'].fillna('') + '_' + patterns['babycategory'].fillna('')
patterns['final_category'] = patterns['final_category'].str.strip('_')  # Remove leading underscore

patterns = patterns.drop(columns = ['supercategory', 'category', 'subcategory', 'babycategory'])

# final_category will be target-encoded after the dataset has been split into train/test/validate groups

In [11]:
patterns.tail()

,yarn_weight,is_clothing,created_at,uk_terminology,us_terminology,yardage,attributes_2-at-a-time,attributes_3-4-sleeve,attributes_3-dimensional,attributes_Intarsia,...,yarn_fiber_Wool,yarn_fiber_Yak,price_usd,days_since_previous_pattern,knit,weight,season_spring,season_summer,season_winter,final_category
801683,-1.545488,True,2023-07-19 05:24:58+00:00,False,False,0.680108,False,False,False,False,...,True,False,0.014430,0.220720,1,1.000303,False,True,False,Accessories_Feet / Legs_Socks_Mid-calf
801684,-1.072667,True,2022-12-02 08:20:23+00:00,False,False,0.209994,False,False,False,False,...,False,False,-0.462688,0.066870,1,1.000200,False,False,True,Accessories_Feet / Legs_Socks_Mid-calf
801686,-0.118863,True,2024-10-22 05:51:45+00:00,False,False,0.632024,False,False,False,False,...,True,False,-0.398464,0.437605,1,1.000614,False,False,False,Accessories_Feet / Legs_Socks_Mid-calf
801687,-0.118863,True,2022-03-08 11:21:23+00:00,False,False,0.761987,False,False,False,False,...,True,False,0.422640,-0.204388,1,1.000101,True,False,False,Clothing_Sweater_Cardigan
801688,-0.118863,True,2023-10-04 19:07:01+00:00,False,False,0.793284,False,False,False,False,...,False,False,0.820946,0.264382,1,1.000344,False,False,False,Clothing_Sweater_Pullover


## Hypothesis Testing

In [12]:
model = smf.ols(
	formula = 'projects_per_day ~ price_usd * knit',
	data = patterns
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       projects_per_day   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.065
Method:                 Least Squares   F-statistic:                 1.289e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        14:08:49   Log-Likelihood:            -1.7595e+05
No. Observations:              554747   AIC:                         3.519e+05
Df Residuals:                  554743   BIC:                         3.520e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.1219      0.001   -171.

For the `price_usd:knit` interaction, the coefficient of 0.0713 with a p-value of 0.000 substantiates the hypothesis that knit patterns sell slightly better than crochet patterns as price increases.

In [13]:
# Get only most expensive 25% of patterns - this is equivalent to $6+ pre-scaling
expensive_patterns = patterns[patterns['price_usd'] >= 0.697828]

model = smf.ols(
	formula = 'projects_per_day ~ price_usd * knit',
	data = expensive_patterns
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       projects_per_day   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     1570.
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        14:08:50   Log-Likelihood:                -96306.
No. Observations:              125986   AIC:                         1.926e+05
Df Residuals:                  125982   BIC:                         1.927e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.0983      0.008    -13.

Looking at only the most expensive 25% of patterns ($6+ USD), there is a stronger correlation between price increases and sales performance for knit patterns vs crochet. Within the more expensive segment of the market, the coefficient is 0.1070 with a p-value of 0.000.

In [14]:
# Hypothesis testing for impact of wool fiber
model = smf.ols(
	formula = 'projects_per_day ~ yarn_fiber_Wool',
	data = patterns
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       projects_per_day   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     3552.
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        14:08:50   Log-Likelihood:            -1.9287e+05
No. Observations:              554747   AIC:                         3.858e+05
Df Residuals:                  554745   BIC:                         3.858e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

The coefficient of 0.0702 with a p-value of 0.000 indicates that use of wool yarn in a pattern does produce a slight increase in pattern sales vs non-wool yarns.

## Split Into Training, Testing, & Validation Sets

In [15]:
patterns = patterns.sort_values('created_at').reset_index(drop = True)

train = patterns[patterns['created_at'].dt.date < pd.to_datetime('2025-3-1').date()]
w = train['weight']
test = patterns[(patterns['created_at'].dt.date >= pd.to_datetime('2025-3-1').date()) & (patterns['created_at'].dt.date <= pd.to_datetime('2025-8-31').date())]
val = patterns[(patterns['created_at'].dt.date >= pd.to_datetime('2025-9-1').date()) & (patterns['created_at'].dt.date <= pd.to_datetime('2026-2-28').date())]

In [16]:
X_train = train.drop(columns= ['projects_per_day', 'created_at', 'weight'])
y_train = train['projects_per_day']

X_test = test.drop(columns= ['projects_per_day', 'created_at', 'weight'])
y_test = test['projects_per_day']

X_val = val.drop(columns= ['projects_per_day', 'created_at', 'weight'])
y_val = val['projects_per_day']

## Convert final_category to Target Encoding

In [17]:
# Convert final_category to target encoding
encoder = ce.CatBoostEncoder(cols = ['final_category'])
encoder.fit(X_train['final_category'], y_train)

X_train['final_category'] = encoder.transform(X_train['final_category'])
X_test['final_category'] = encoder.transform(X_test['final_category'])
X_val['final_category'] = encoder.transform(X_val['final_category'])

## Feature Selection

In [18]:
X_train.info(verbose = True)

<class 'pandas.DataFrame'>
RangeIndex: 512199 entries, 0 to 512198
Data columns (total 301 columns):
 #    Column                                   Dtype  
---   ------                                   -----  
 0    yarn_weight                              float64
 1    is_clothing                              bool   
 2    uk_terminology                           bool   
 3    us_terminology                           bool   
 4    yardage                                  float64
 5    attributes_2-at-a-time                   bool   
 6    attributes_3-4-sleeve                    bool   
 7    attributes_3-dimensional                 bool   
 8    attributes_Intarsia                      bool   
 9    attributes_Shetland                      bool   
 10   attributes_adaptive                      bool   
 11   attributes_adult                         bool   
 12   attributes_afterthought-heel             bool   
 13   attributes_afterthought-pocket           bool   
 14   attributes_al

### Polynomial Linear Regression

In [19]:
numeric_cols = [
	'yarn_weight',
	'yardage',
	'price_usd',
	'days_since_previous_pattern',
	'final_category'
]

preprocess = ColumnTransformer(
	transformers = [
		('poly', PolynomialFeatures(degree = 2, include_bias = False), numeric_cols),
		('pass', 'passthrough', [c for c in X_train.columns if c not in numeric_cols])
	]
)

In [20]:
pipe = Pipeline([
	('prep', preprocess),
	('model', ElasticNet())
])

pipe.fit(X_train, y_train, model__sample_weight = w)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('poly', ...), ('pass', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains

In [21]:
pipe.score(X_test, y_test)

-0.04169546677813685

Performs very slightly better (less bad) than ElasticNet regression without polynomial features.

### Random Forest

In [22]:
# Parameters have been reduced to prevent kernel crashing
rf = RandomForestRegressor(
    n_estimators=50,      # Reduce from default 100
    max_depth=15,         # Limit tree depth
    n_jobs=1,             # Use single core instead of all cores
    verbose=0,
    random_state=42026
)

rf.fit(X_train, y_train, sample_weight = w)

rf.score(X_test, y_test)

0.19144260423515336

This one is performing slightly better than a null model! Let's check the feature importances.

In [23]:
# Just the top ones
for score, name in zip(rf.feature_importances_, X_train.columns):
	if score >= 0.01:
		print(round(score, 2), name)

0.02 yarn_weight
0.05 yardage
0.01 attributes_positive-ease
0.04 attributes_top-down
0.01 attributes_video-tutorial
0.01 pattern_source_type_names_Book
0.06 previously_published_patterns
0.02 yarn_fiber_Merino
0.17 price_usd
0.08 days_since_previous_pattern
0.01 knit
0.05 final_category


In [24]:
# All feature importances
for score, name in zip(rf.feature_importances_, X_train.columns):
	print(round(score, 4), name)

0.0247 yarn_weight
0.0028 is_clothing
0.0009 uk_terminology
0.0005 us_terminology
0.0485 yardage
0.0005 attributes_2-at-a-time
0.0009 attributes_3-4-sleeve
0.0014 attributes_3-dimensional
0.0012 attributes_Intarsia
0.0019 attributes_Shetland
0.0001 attributes_adaptive
0.0042 attributes_adult
0.0016 attributes_afterthought-heel
0.0013 attributes_afterthought-pocket
0.0011 attributes_aline
0.0019 attributes_amigurumi
0.0003 attributes_andean
0.0005 attributes_appliqued
0.0006 attributes_aran
0.0033 attributes_asymmetric
0.0027 attributes_baby
0.0007 attributes_backfastening
0.0007 attributes_ballet-neck
0.0002 attributes_bavarian
0.0028 attributes_beads
0.0052 attributes_bias
0.0006 attributes_boat-neck
0.0025 attributes_bobble-or-popcorn
0.003 attributes_bottom-up
0.0004 attributes_box-pleats
0.0014 attributes_bracelet-sleeve
0.0006 attributes_braids-plaiting
0.0027 attributes_brioche-tuck
0.0 attributes_broomstick
0.0 attributes_bruges
0.0 attributes_bullion
0.0016 attributes_buttoned
